# Ejecutar las pruebas de Maderas MyM en Google Colab
1. En el panel de archivos (icono de carpeta, a la izquierda) suba **MaderasMyM_Pruebas_Proyecto_Cucumber.zip**.
2. Ejecute las celdas en orden (botón ▶). La celda 3 demora varios minutos.
3. Al final se descarga **resultados_pruebas.zip**: envíelo al chat para armar el informe.

In [ ]:
# 1) Instalar Java 8, Maven y Google Chrome (con su chromedriver)
!apt-get -qq update
!apt-get -qq install -y openjdk-8-jdk-headless maven unzip libnss3 libgbm1 libasound2 libxkbcommon0 libatk-bridge2.0-0 libcups2 > /dev/null
!wget -q -O /tmp/cft.json https://googlechromelabs.github.io/chrome-for-testing/last-known-good-versions-with-downloads.json
import json, subprocess
d = json.load(open('/tmp/cft.json'))['channels']['Stable']['downloads']
u_chrome = [x['url'] for x in d['chrome'] if x['platform']=='linux64'][0]
u_drv = [x['url'] for x in d['chromedriver'] if x['platform']=='linux64'][0]
!wget -q -O /tmp/chrome.zip {u_chrome}
!wget -q -O /tmp/drv.zip {u_drv}
!rm -rf /opt/cft && mkdir -p /opt/cft && unzip -q /tmp/chrome.zip -d /opt/cft && unzip -q /tmp/drv.zip -d /opt/cft
!ls /opt/cft


In [ ]:
# 2) Descomprimir el proyecto
!rm -rf /content/p && mkdir /content/p && unzip -q /content/MaderasMyM_Pruebas_Proyecto_Cucumber.zip -d /content/p
!ls /content/p/MaderasMyM_Pruebas


In [ ]:
# 3) Ejecutar los 15 Runner (varios minutos)
import os
os.environ['HEADLESS'] = 'true'
os.environ['CHROMEDRIVER_PATH'] = '/opt/cft/chromedriver-linux64/chromedriver'
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
# Chrome debe poder encontrarse: se indica su ruta al chromedriver
os.environ['CHROME_BIN'] = '/opt/cft/chrome-linux64/chrome'
!ln -sf /opt/cft/chrome-linux64/chrome /usr/local/bin/google-chrome
!cd /content/p/MaderasMyM_Pruebas && /opt/cft/chrome-linux64/chrome --version > entorno.txt && echo "fecha: $(date -u)" >> entorno.txt && mvn -B clean verify 2>&1 | tail -n 60


In [ ]:
# 4) Procesar resultados y empaquetar
!pip -q install openpyxl
!cd /content/p/MaderasMyM_Pruebas && python herramientas/procesar_resultados.py --resumen-json resumen_resultados.json | tee procesamiento.txt
!cd /content/p/MaderasMyM_Pruebas && zip -qr /content/resultados_pruebas.zip evidencias target/cucumber-json target/cucumber-reports target/cucumber-report-html target/cucumber-results planilla resumen_resultados.json entorno.txt procesamiento.txt
from google.colab import files
files.download('/content/resultados_pruebas.zip')
